# V3 Sweep Monitor

Read-only status for Reuters, ROCStories, and WebText. Training runs in a separate kernel. This notebook does not start, resume, interrupt, or modify training. Run Cell 2 to refresh the snapshot.

In [ ]:
import json
from datetime import datetime, timezone
from pathlib import Path

ROOT = Path('/home/jovyan/cisco-cpt-1a/nlp_v3_remaining_20261105')
ATTEMPT = ROOT / 'continuations/20261105T142910Z'
if not ROOT.is_dir():
    raise RuntimeError('Open this notebook on the Prayogashala server.')

process = json.loads((ATTEMPT / 'process.json').read_text())
try:
    command = (Path('/proc') / str(process['pid']) / 'cmdline').read_text()
    running = 'run_kubeflow_sweep.py' in command
except FileNotFoundError:
    running = False
print('Checked:', datetime.now(timezone.utc).isoformat())
print('Launcher running:', running, '| PID:', process['pid'])
print('Entire sweep complete:', (ROOT / 'run/COMPLETED.json').is_file())
exit_file = ATTEMPT / 'exit.json'
if exit_file.is_file():
    print('Exit:', json.loads(exit_file.read_text()))

for corpus, folder, expected in (
    ('Reuters', 'reuters_neural', 45),
    ('ROCStories', 'roc-stories', 15),
    ('WebText', 'webtext-nn', 15),
):
    run_root = ROOT / 'run/full/runs' / folder
    saved = len(list(run_root.glob('train_*pct/models/*/*.pt')))
    print(f'{corpus}: {saved}/{expected} final checkpoint files saved')
progress_file = ROOT / 'run/progress.json'
if progress_file.is_file():
    print('Completed corpora:', json.loads(progress_file.read_text())['completed_corpora'])

print()
print('Recent log output:')
with (ATTEMPT / 'continuation.log').open('rb') as handle:
    handle.seek(0, 2)
    handle.seek(max(0, handle.tell() - 6000))
    print(handle.read().decode('utf-8', errors='replace'))